# GamePulse — Moteur de recommandation de jeux Steam

Ce notebook implémente un pipeline de recherche sémantique sur une base de **114 000 jeux Steam**, combinant quatre techniques complémentaires :

| Étape | Technique | Rôle |
|-------|-----------|------|
| 1 | **Query Expansion** (Mistral) | Enrichit la requête utilisateur en 3 sous-requêtes optimisées |
| 2 | **Multi-Query RRF** | Fusionne les résultats de plusieurs requêtes via Reciprocal Rank Fusion |
| 3 | **ChromaDB** (bi-encoder) | Recherche vectorielle sur les descriptions de jeux |
| 4 | **Cross-Encoder Reranking** | Reclasse finement les candidats par pertinence |


## 1. Initialisation — Imports & clés API

Chargement des bibliothèques et des clés API depuis le fichier `.env`. Deux clés sont nécessaires :
- **`STEAM_API_KEY`** — accès aux données Steam
- **`MISTRAL_API_KEY`** — query expansion via le LLM Mistral

In [53]:
import os
import json
import requests
from collections import defaultdict
from dotenv import load_dotenv

load_dotenv('../.env')

STEAM_KEY   = os.getenv("STEAM_API_KEY")
MISTRAL_KEY = os.getenv("MISTRAL_API_KEY")

print("✅ Clé Steam   :", "OK" if STEAM_KEY   else "❌ Introuvable")
print("✅ Clé Mistral :", "OK" if MISTRAL_KEY else "❌ Introuvable")

✅ Clé Steam   : OK
✅ Clé Mistral : OK


## 2. Connexion à ChromaDB

Connexion à la base vectorielle persistante (`chroma_db_full_v2`) contenant les embeddings des 114 000 jeux Steam.
Le modèle d'embedding utilisé est `all-MiniLM-L6-v2` (bi-encoder léger, ~80 MB).

> On utilise `get_collection` et non `get_or_create_collection` pour éviter tout écrasement accidentel de la base.

In [54]:
import chromadb
from chromadb.utils import embedding_functions

chroma_client = chromadb.PersistentClient(path="data/chroma_db_full_v2")

sentence_transformer_ef = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

collection = chroma_client.get_collection(
    name="gamepulse_collection",
    embedding_function=sentence_transformer_ef
)

print(f"✅ ChromaDB prêt — {collection.count():,} jeux indexés")

✅ ChromaDB prêt — 114,161 jeux indexés


## 3. Multi-Query + Reciprocal Rank Fusion (RRF)

Un seul vecteur de requête tend à écraser les critères rares (ex : *espace*, *pixel art*). Pour contourner ce problème, on lance **plusieurs sous-requêtes en parallèle** et on fusionne leurs classements via la formule RRF :

$$\text{score\_RRF}(d) = \sum_{q} \frac{1}{k + \text{rank}_q(d)}$$

Chaque document gagne des points selon sa position dans chaque classement. La constante `k=60` atténue l'avantage des premiers rangs.

In [ ]:
def multi_query_rrf_search(collection, queries: list[str], n_per_query: int = 15, top_k: int = 5, k: int = 60):
    """
    Lance plusieurs requêtes sur ChromaDB et fusionne les classements via RRF.

    Args:
        collection  : Collection ChromaDB
        queries     : Liste de requêtes complémentaires
        n_per_query : Candidats récupérés par requête
        top_k       : Nombre de résultats finaux
        k           : Constante RRF (60 par défaut)

    Returns:
        Liste de tuples (metadata, document, rrf_score)
    """
    rrf_scores = defaultdict(float)
    game_store = {}  # doc_id → (metadata, document)

    for query in queries:
        results = collection.query(query_texts=[query], n_results=n_per_query)
        for rank, (doc_id, metadata, document) in enumerate(
            zip(results['ids'][0], results['metadatas'][0], results['documents'][0])
        ):
            rrf_scores[doc_id] += 1.0 / (k + rank + 1)
            game_store[doc_id]  = (metadata, document)

    sorted_ids = sorted(rrf_scores, key=lambda x: rrf_scores[x], reverse=True)[:top_k]
    return [(game_store[i][0], game_store[i][1], rrf_scores[i]) for i in sorted_ids]

## 4. Query Expansion via Mistral

L'utilisateur formule une requête en langage naturel (français ou anglais), souvent vague. Mistral la décompose automatiquement en **3 sous-requêtes anglaises** ciblées, chacune mettant en avant un aspect différent :
- la mécanique principale
- le genre ou l'univers
- le critère le plus rare/spécifique

Ces sous-requêtes sont ensuite passées à la fonction RRF ci-dessus.

In [ ]:
def expand_query_with_mistral(user_query: str, mistral_api_key: str, n_queries: int = 3) -> list[str]:
    """
    Utilise Mistral pour générer plusieurs sous-requêtes optimisées pour ChromaDB.

    Args:
        user_query      : Requête utilisateur brute (FR ou EN)
        mistral_api_key : Clé API Mistral
        n_queries       : Nombre de sous-requêtes à générer

    Returns:
        Liste de {n_queries} requêtes anglaises optimisées
    """
    prompt = f"""You are a Steam game search optimization expert.
A user wants to find games matching: "{user_query}"

Generate exactly {n_queries} distinct search queries in English to find relevant games.
Rules:
- Each query must emphasize a DIFFERENT key aspect of the request
- Use Steam-specific vocabulary (tags, genres, mechanics)
- Be concise (5-10 words per query)
- Isolate the rarest/most specific criteria in at least one query

Return ONLY a valid JSON array of {n_queries} strings. No preamble, no explanation.
Example: ["query one here", "query two here", "query three here"]"""

    try:
        response = requests.post(
            "https://api.mistral.ai/v1/chat/completions",
            headers={
                "Authorization": f"Bearer {mistral_api_key}",
                "Content-Type": "application/json"
            },
            json={
                "model": "mistral-small-latest",
                "messages": [{"role": "user", "content": prompt}],
                "temperature": 0.3,
                "max_tokens": 200
            },
            timeout=15
        )
        response.raise_for_status()

        raw = response.json()["choices"][0]["message"]["content"].strip()
        raw = raw.replace("```json", "").replace("```", "").strip()
        queries = json.loads(raw)

        if isinstance(queries, list):
            return queries[:n_queries]
        return list(queries.values())[:n_queries]

    except Exception as e:
        print(f"⚠️ Mistral expansion échouée ({e}), fallback sur requête brute.")
        return [user_query]

## 5. Chargement du Cross-Encoder

Le bi-encoder (`all-MiniLM-L6-v2`) encode la requête et les documents **séparément**, ce qui limite la précision. Le cross-encoder (`ms-marco-MiniLM-L-6-v2`) analyse chaque paire `(requête, document)` **ensemble**, capturant les interactions fines entre les deux.

Cette étape est plus coûteuse en calcul, donc on l'applique uniquement sur les **20 meilleurs candidats** issus du RRF.

In [55]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print("✅ Cross-encoder chargé")

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Cross-encoder chargé


## 6. Pipeline complet — `gamepulse_search`

La fonction `gamepulse_search` orchestre les étapes précédentes dans l'ordre optimal :

```
Requête utilisateur (FR/EN)
       ↓
  Mistral → 3 sous-requêtes enrichies
       ↓
  Multi-Query RRF → 20 candidats fusionnés
       ↓
  Cross-Encoder → reranking final
       ↓
  Top K résultats
```

In [ ]:
def gamepulse_search(
    user_query: str,
    collection,
    mistral_api_key: str = None,
    use_reranker: bool = True,
    top_k: int = 5,
    verbose: bool = True
) -> list[dict]:
    """
    Pipeline de recommandation GamePulse.

    Args:
        user_query      : Requête naturelle (FR ou EN)
        collection      : Collection ChromaDB
        mistral_api_key : Clé Mistral pour la query expansion
        use_reranker    : Activer le cross-encoder reranking
        top_k           : Nombre de résultats finaux
        verbose         : Afficher les logs intermédiaires

    Returns:
        Liste de dicts avec name, price, tags, score, description
    """

    # Étape 1 : Query Expansion via Mistral
    if mistral_api_key:
        if verbose: print("🤖 [Axe 2] Query expansion via Mistral...")
        queries = expand_query_with_mistral(user_query, mistral_api_key, n_queries=3)
        if verbose:
            for i, q in enumerate(queries, 1): print(f"   {i}. {q}")
    else:
        if verbose: print("⚠️  Mistral non configuré — utilisation de la requête brute")
        queries = [user_query]

    # Étape 2 : Multi-Query RRF (wide retrieval)
    if verbose: print("\n🔬 [Axes 1+3] Recherche multi-query avec filtres hybrides...")

    n_candidates = max(top_k * 4, 20)
    rrf_scores = defaultdict(float)
    game_store  = {}
    K_RRF = 60

    for query in queries:
        results = collection.query(query_texts=[query], n_results=n_candidates)
        for rank, (doc_id, metadata, document) in enumerate(
            zip(results['ids'][0], results['metadatas'][0], results['documents'][0])
        ):
            rrf_scores[doc_id] += 1.0 / (K_RRF + rank + 1)
            game_store[doc_id]  = (metadata, document)

    top_candidates_ids = sorted(rrf_scores, key=lambda x: rrf_scores[x], reverse=True)[:n_candidates]
    candidates_meta    = [game_store[i][0] for i in top_candidates_ids]
    candidates_docs    = [game_store[i][1] for i in top_candidates_ids]

    if verbose: print(f"   → {len(top_candidates_ids)} candidats après RRF")

    # Étape 3 : Cross-Encoder Reranking
    if use_reranker and 'reranker' in globals():
        if verbose: print("\n🏆 [Axe 4] Cross-encoder reranking...")
        pairs  = [(user_query, doc) for doc in candidates_docs]
        scores = reranker.predict(pairs)

        ranked = sorted(
            zip(scores, candidates_meta, candidates_docs),
            key=lambda x: x[0], reverse=True
        )[:top_k]

        final_results = [
            {"name": m['Name'], "price": m['Price'],
             "tags": m.get('Tags', 'N/A'), "score": round(float(s), 3),
             "description": d.split("DESC: ")[-1][:200]}
            for s, m, d in ranked
        ]
    else:
        final_results = [
            {"name": game_store[doc_id][0]['Name'],
             "price": game_store[doc_id][0]['Price'],
             "tags": game_store[doc_id][0].get('Tags', 'N/A'),
             "score": round(rrf_scores[doc_id], 4),
             "description": game_store[doc_id][1].split("DESC: ")[-1][:200]}
            for doc_id in top_candidates_ids[:top_k]
        ]

    return final_results


print("✅ Pipeline gamepulse_search() défini")

## 7. Test du pipeline

Lance une recherche complète avec une requête en français. Modifie `user_query` pour tester d'autres recherches.

In [56]:
results = gamepulse_search(
    user_query      = "Je cherche un jeu de survie sur une île",
    collection      = collection,
    mistral_api_key = MISTRAL_KEY,
    use_reranker    = True,
    top_k           = 5,
    verbose         = True
)

print("\n" + "=" * 60)
print("🎮 RÉSULTATS FINAUX")
print("=" * 60)
for i, game in enumerate(results, 1):
    print(f"\n{i}. {game['name']}")
    print(f"   Prix  : {game['price']}")
    print(f"   Tags  : {str(game['tags'])[:100]}")
    print(f"   Score : {game['score']}")
    print(f"   Desc  : {game['description'][:150]}...")

🤖 [Axe 2] Query expansion via Mistral...
   1. survival island simulation Steam
   2. hardcore survival sandbox island games
   3. island survival roguelike Steam

🔬 [Axes 1+3] Recherche multi-query avec filtres hybrides...
   → 20 candidats après RRF

🏆 [Axe 4] Cross-encoder reranking...

🎮 RÉSULTATS FINAUX

1. Archipelago: Island Survival
   Prix  : 4.99
   Tags  : N/A
   Score : -11.27
   Desc  : Archipelago is a unique survival simulator with sandbox, trading, and RPG elements set in a medieval fantasy world. Your ship crashes, and you find yo...

2. Island
   Prix  : 4.99
   Tags  : N/A
   Score : -11.342
   Desc  : Island is a TPP survival running game which is existing on a fantasy island.But watch out!There are several different kinds of traps on those island. ...

3. Accursed Isle: Survival
   Prix  : 1.99
   Tags  : N/A
   Score : -11.344
   Desc  : On a deserted island, you are trying to survive amidst unknown creatures and dangers. Facing hunger, thirst, and deadly threats,